# LoRA rank-threshold study: Kaggle runner (2×T4)

Settings: **Accelerator: GPU T4 ×2**, **Internet: on**, and a Kaggle secret named `HF_TOKEN` with write access.

Everything resumes. Finished runs and the registry are mirrored to a private Hugging Face repo, so a new session picks up where the last one stopped. Run order: `lr_cal` → edit LRs in `grid_*.yaml` → `grid_sql`, `grid_facts` → oracle → `ablations`.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

REPO_URL = "https://github.com/nayyirahsan/lora-rank-threshold.git"
HUB_REPO = "<your-hf-username>/lorathresh-ckpts"  # EDIT: private model repo for checkpoints + registry

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HUB_REPO"] = HUB_REPO
os.environ["PYTHONUNBUFFERED"] = "1"  # background shard logs update live, so the progress cell shows real steps

In [ ]:
%%bash
set -e
[ -d LoRA ] || git clone "$REPO_URL" LoRA
cd LoRA && git pull --ff-only
pip install -q -e ".[gpu]"
python -c "from huggingface_hub import create_repo; create_repo('$HUB_REPO', private=True, exist_ok=True)"
nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Gate: measure T4 throughput and peak memory before spending quota

These are the two most memory-hungry configs: full FT on SQL, which has the longest sequences and 152k-vocab logits, and LoRA r=64 for comparison. They write to a scratch registry so they never mix with study results. Compare `train_tokens_per_second` with the plan's estimate. If full FT runs out of memory, rerun with `--micro-batch-size 8` (same math, less memory) and use that flag for the grids.

In [ ]:
%%bash
cd LoRA
python -m lorathresh.train --task sql --n 2000 --method full --lr 3e-5 --epochs 1 --n-eval 200 --max-steps 60 --registry /kaggle/working/gate.jsonl --output-dir /kaggle/working/gate_ckpt
python -m lorathresh.train --task sql --n 2000 --method lora --rank 64 --lr 3e-4 --epochs 1 --n-eval 200 --max-steps 60 --registry /kaggle/working/gate.jsonl --output-dir /kaggle/working/gate_ckpt
python - <<'EOF'
import json
for line in open('/kaggle/working/gate.jsonl'):
    r = json.loads(line); m = r['metrics']
    print(r['config']['method'], r['config']['rank'], {k: round(m[k], 2) for k in ('train_tokens_per_second', 'peak_memory_gb', 'eval_seconds')})
EOF

## 2. Launch a grid on both GPUs

Set `GRID`, then run the cell. The shards run in the background, so the notebook stays usable. Kaggle stops background processes when the session ends. Both shards mirror to the Hub after every run, so just relaunch in the next session.

In [ ]:
os.environ["GRID"] = "lr_cal"        # lr_cal | grid_sql | grid_facts | ablations
os.environ["MICRO"] = ""             # e.g. "--micro-batch-size 8" if the gate ran out of memory

In [ ]:
%%bash
cd LoRA
python scripts/run_grid.py configs/$GRID.yaml --dry-run --push-repo $HUB_REPO | head -3
CUDA_VISIBLE_DEVICES=0 nohup python scripts/run_grid.py configs/$GRID.yaml --shard 0/2 --push-repo $HUB_REPO $MICRO > shard0.log 2>&1 &
CUDA_VISIBLE_DEVICES=1 nohup python scripts/run_grid.py configs/$GRID.yaml --shard 1/2 --push-repo $HUB_REPO $MICRO > shard1.log 2>&1 &
sleep 5; echo launched

In [ ]:
%%bash
# Re-run any time to check progress.
cd LoRA
for s in 0 1; do echo "== shard $s"; grep -E '^===|^step|Traceback|Error' shard$s.log | tail -3; done
echo "== finished runs: $(wc -l < results/runs.jsonl 2>/dev/null || echo 0)"
nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv,noheader

## 3. Oracle sweep for every full-FT run (after the main grids)

Checkpoints come from the Hub if this session doesn't have them locally.

In [ ]:
%%bash
cd LoRA
python - <<'EOF' > ft_runs.txt
from lorathresh.registry import Registry
for r in Registry('results/runs.jsonl').rows():
    if r['config'].get('method') == 'full' and r['config'].get('max_steps') is None:
        print(r['run_id'])
EOF
echo "$(wc -l < ft_runs.txt) full-FT runs"
while read rid; do python scripts/run_oracle.py --ft-run-id $rid --push-repo $HUB_REPO --svd-device cuda; done < ft_runs.txt

## 4. Aggregate

Rebuilds `results/r_star.md` and `figures/*.png` from whatever has finished. Download them, or commit them to the repo.

In [ ]:
%%bash
cd LoRA
python scripts/aggregate.py --registry results/runs.jsonl --out results --figures figures
cp -r results figures /kaggle/working/